# SQL Worksheet — Week3

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Filtered Student Risk Profile
Join students to attendance and class data, filter to students whose names start with `A` and whose attendance is Late or Absent, then group by student and topic. Return issue count, class date range, and a null-safe topic label.

In [0]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    COUNT(f.attendance_id) AS issue_count,
    MIN(c.class_date) AS first_issue_date,
    MAX(c.class_date) AS last_issue_date
FROM rivadataplatform.dataproduct.dim_student AS s
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
WHERE s.student_name LIKE 'A%'
    AND f.attendance_status IN ('Late', 'Absent')
GROUP BY s.student_id, s.student_name, COALESCE(c.topic, 'Topic not assigned')
ORDER BY issue_count DESC, s.student_name;

## Question 2 — Date-Range Attendance Detail
Join attendance to students, classes, batches, and `dim_date`. Return records whose class date falls between the batch start and end dates, showing student, batch, topic, calendar day, and status. Exclude records with a null status and sort chronologically.

In [0]:
SELECT
    f.attendance_id,
    s.student_name,
    b.batch_name,
    c.class_date,
    COALESCE(d.day_name, c.class_day) AS day_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    f.attendance_status
FROM rivadataplatform.dataproduct.fact_attendance AS f
JOIN rivadataplatform.dataproduct.dim_student AS s
    ON s.student_key = f.student_key
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
WHERE c.class_date BETWEEN b.start_date AND b.end_date
    AND NULLIF(f.attendance_status, '') IS NOT NULL
ORDER BY c.class_date, s.student_name;

## Question 3 — Conditional Batch Scorecard
Join attendance to batch and date dimensions and return one row per batch and calendar month. Calculate Present, Late, Absent, total records, and distinct students. Use conditional aggregation and keep only months containing at least one Absent record.

In [0]:
SELECT
    b.batch_id,
    b.batch_name,
    COALESCE(d.year, EXTRACT(YEAR FROM f.joined_at)) AS attendance_year,
    COALESCE(d.month, EXTRACT(MONTH FROM f.joined_at)) AS attendance_month,
    COUNT(DISTINCT f.student_key) AS distinct_students,
    COUNT(f.attendance_id) AS total_records,
    SUM(CASE WHEN f.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count,
    SUM(CASE WHEN f.attendance_status = 'Late' THEN 1 ELSE 0 END) AS late_count,
    SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS absent_count
FROM rivadataplatform.dataproduct.fact_attendance AS f
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
GROUP BY b.batch_id, b.batch_name,
    COALESCE(d.year, EXTRACT(YEAR FROM f.joined_at)),
    COALESCE(d.month, EXTRACT(MONTH FROM f.joined_at))
HAVING SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) > 0
ORDER BY attendance_year, attendance_month;

## Question 4 — Class Coverage Including Empty Classes
Use `dim_class` as the driving table and left join attendance, students, batch, and date dimensions. Group by class and return class metadata, distinct students, total records, and average `attendance_count`, showing zero/null-safe values for classes without attendance.

In [0]:
SELECT
    c.class_id,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    b.batch_name,
    COALESCE(d.day_name, c.class_day) AS day_name,
    COUNT(DISTINCT f.student_key) AS distinct_students,
    COUNT(f.attendance_id) AS attendance_records,
    COALESCE(AVG(f.attendance_count), 0) AS average_attendance_count
FROM rivadataplatform.dataproduct.dim_class AS c
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.class_key = c.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_id = c.batch_id
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
GROUP BY c.class_id, COALESCE(c.topic, 'Topic not assigned'), b.batch_name,
    COALESCE(d.day_name, c.class_day)
ORDER BY c.class_id;

## Question 5 — Average Attendance by Topic and Status
Join attendance to class and batch dimensions. Group by batch, null-safe topic, and attendance status, and calculate average `attendance_count`, total records, and distinct students. Exclude null/blank statuses and sort by average descending.

In [0]:
SELECT
    b.batch_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    f.attendance_status,
    AVG(f.attendance_count) AS average_attendance_count,
    COUNT(f.attendance_id) AS attendance_records,
    COUNT(DISTINCT f.student_key) AS distinct_students
FROM rivadataplatform.dataproduct.fact_attendance AS f
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
WHERE NULLIF(f.attendance_status, '') IS NOT NULL
GROUP BY b.batch_name, COALESCE(c.topic, 'Topic not assigned'), f.attendance_status
ORDER BY average_attendance_count DESC, topic;

## Question 6 — High-Volume Students With Profile Gaps
Use a `LEFT JOIN` from students through attendance, classes, and batches. Group by student and batch, then return students with at least two records, including Present count, issue count, and a null-safe phone label. Order by issue count and total records.

In [0]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(b.batch_name, 'No batch') AS batch_name,
    COALESCE(NULLIF(s.phone_no, ''), 'Phone missing') AS phone_status,
    COUNT(f.attendance_id) AS attendance_records,
    SUM(CASE WHEN f.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count,
    SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) AS issue_count
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY s.student_id, s.student_name,
    COALESCE(b.batch_name, 'No batch'),
    COALESCE(NULLIF(s.phone_no, ''), 'Phone missing')
HAVING COUNT(f.attendance_id) >= 2
ORDER BY issue_count DESC, attendance_records DESC;